In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score,  RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from scipy.stats import loguniform

#!pip install xlrd 
#!pip install category_encoders
import category_encoders as ce



In [ ]:
df = pd.read_excel("CW2/TrainDataset2025.xls")
df.head()

In [ ]:
df.replace(999, pd.NA, inplace=True)

In [ ]:
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0]
missing_cols

In [ ]:
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape

In [ ]:
# Iterative imputation

df_imputed = df_dropped.copy()

# Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


# Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
# Separate target
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

# -----------------------------
# Build X, y
# -----------------------------
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


X = pd.get_dummies(X, columns=cat_features, drop_first=True)
#one hot encoding categorical features

#print("Number of numeric features:", len(num_features))
#print("Categorical features:", cat_features)

# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


In [ ]:
#normalise features 
from tensorflow import keras
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to numpy arrays
X_train = np.array(X_train_scaled)
X_test = np.array(X_test_scaled)
y_train = np.array(y_train)
y_test = np.array(y_test)

# ================================================================
# 9. ANN MODEL (MLP)
# ================================================================
model = keras.models.Sequential([
    keras.layers.Dense(64, input_dim=X_train.shape[1], activation="relu"),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")     # Binary classification
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()



In [ ]:
#train model 
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# ================================================================
# 11. EVALUATION
# ================================================================
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\n=== ACCURACY ===")
print(accuracy_score(y_test, y_pred))

print("\n=== ROC AUC ===")
print(roc_auc_score(y_test, y_pred_prob))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))